# Heikin-Ashi Trend Strategy (Advanced)

Trend-following strategy using Heikin-Ashi candles
and moving average crossover on HA close.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from strategies.advanced.heikin_ashi_trend import HeikinAshiTrendStrategy

In [ ]:
TRANSACTION_COST = 0.0015


def evaluate_strategy(df):
    df = df.copy()

    df["returns"] = df["close"].pct_change()
    df["strategy_returns"] = df["signal"].shift(1) * df["returns"]

    df["strategy_returns"] -= np.where(
        df["signal"] != 0, TRANSACTION_COST, 0
    )

    df["cum_market"] = (1 + df["returns"]).cumprod()
    df["cum_strategy"] = (1 + df["strategy_returns"]).cumprod()

    df["drawdown"] = (
        df["cum_strategy"] / df["cum_strategy"].cummax() - 1
    )

    return df


def plot_results(df, title):
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    ax1.plot(df.index, df["strategy_returns"])
    ax1.set_title(f"{title} – Strategy Returns")

    ax2.plot(df.index, df["cum_market"], label="Market")
    ax2.plot(df.index, df["cum_strategy"], label="Strategy")
    ax2.legend()
    ax2.set_title(f"{title} – Cumulative Returns")

    ax3.plot(df.index, df["drawdown"], color="red")
    ax3.set_title(f"{title} – Drawdown")

    plt.tight_layout()
    plt.show()


In [ ]:
def run_experiment(csv_path, title, short_window, long_window):
    df = pd.read_csv(csv_path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df.set_index("datetime", inplace=True)

    strategy = HeikinAshiTrendStrategy(
        short_window=short_window,
        long_window=long_window,
    )

    df = strategy.run(df)
    df = evaluate_strategy(df)

    plot_results(df, title)
    return df

In [ ]:
btc_30m = run_experiment(
    "data/processed/btc_30m.csv",
    "BTC 30m – Heikin Ashi Trend",
    short_window=60,
    long_window=300,
)

eth_30m = run_experiment(
    "data/processed/eth_30m.csv",
    "ETH 30m – Heikin Ashi Trend",
    short_window=60,
    long_window=300,
)

In [ ]:
import os

def save_experiment_summary(
    df,
    strategy_name,
    asset,
    timeframe,
    out_dir="results/csv_outputs"
):
    """
    Save a one-row CSV summarizing strategy performance.
    """

    os.makedirs(out_dir, exist_ok=True)

    total_return = df["cumulative_strategy_returns"].iloc[-1] - 1
    benchmark_return = df["cumulative_market_returns"].iloc[-1] - 1
    max_drawdown = df["drawdown"].min()
    win_rate = (df["strategy_returns"] > 0).mean()
    total_trades = df["signal"].diff().abs().sum() / 2

    row = {
        "strategy": strategy_name,
        "asset": asset,
        "timeframe": timeframe,
        "total_return_pct": round(total_return * 100, 2),
        "benchmark_return_pct": round(benchmark_return * 100, 2),
        "max_drawdown_pct": round(max_drawdown * 100, 2),
        "win_rate_pct": round(win_rate * 100, 2),
        "total_trades": int(total_trades),
    }

    df_out = pd.DataFrame([row])

    out_file = os.path.join(out_dir, f"{strategy_name}.csv")

    if os.path.exists(out_file):
        df_out.to_csv(out_file, mode="a", header=False, index=False)
    else:
        df_out.to_csv(out_file, index=False)

    print(f"✅ Saved summary → {out_file}")


In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="heiken_ashi_trend",
    asset="BTC",
    timeframe="30M"
)

In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="heiken_ashi_trend",
    asset="ETH",
    timeframe="30M"
)